# Introduction


## References

* https://medium.com/@van3ssabandeira/o-famoso-spacy-90afb683b6fe
* [Python para PLN - Disciplinas USP](https://edisciplinas.usp.br/pluginfile.php/6305937/mod_resource/content/0/Aula%2010%20-%20python%20para%20PLN%20-%20spaCy.pdf)

# Requirements

In [1]:
# https://www.tensorflow.org/api_docs/python/tf/keras/layers

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
import emoji
import spacy
import string
import unicodedata
import datetime
import random
from sklearn.utils import shuffle
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Get Data

In [2]:
# transformando e abrindo o arquivo em formato csv
df = pd.read_excel('DateToTestSentiment_20210817.xls.xls')

# Data Prep Functions

* **preprocess_data** 
* **preprocess_text** 
* **tokenizacao**

In [3]:
def preprocess_data(data, 
                    columns,
                    null = True):
    
    df = data[columns]
    
    if null:
        df = df.dropna().reset_index().drop(columns=['index'])
    
    return df


def preprocess_text(text, 
                    remove_stop = True, 
                    stem_words = False, 
                    remove_mentions_hashtags = True
                   ):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # corrects the bug that eliminates letters from words with accents
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    # removes special characters 
    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    #
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()
    
    # removes stopwords with less than two letters
    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)


def tokenizacao(df):
    
    # gets the number of rows and columns from the dataframe
    rows, cols = df.shape

    # creates the dataset column with the vectorized text
    df['token'] = [preprocess_text(df["COMMENT_TEXT"][row]) for row in range(rows)]
    
    return df

# Pipeline Data Prep

   - Tokeniza cada comentário na a coluna 'COMMENT_TEXT' do dataset e depois remove as linhas tokenizadas de tamanho 0

In [4]:
portuguese_stopwords = nltk.corpus.stopwords.words('portuguese')

# escolhe as colunas que usaremos e remove as linhas nulas
df2 = preprocess_data(df, 
                      columns=['KEY','COMMENT_ID','COMMENT_TEXT'],
                      null = True
                     )

# cria coluna com comentários tokenizados
df2 = tokenizacao(df2)

# creates an intermediary dataframe
df3 = df2.copy()

# remove as linhas que a tokenização ficou com tamanho 0 (sem texto)
for n, item in enumerate(df2.token):
    if len(item)==0:
        df3 = df3.drop(n)

# Topicos de classificação

- Cria lista de tópicos em ordem de prioridade que usaremos para a classificação dos comentários
- Criação do dicionário que associa os labels a seu tópico
- Tokeniza o  tópicos para usar na seleção dos labels

In [5]:
topics = ['AQUECIMENTO', 
          'ASSISTÊNCIA TÉCNICA', 
          'ATENDIMENTO', 
          'AUTO FALANTE', 
          'BATERIA', 
          'CAMERA', 
          'CARREGADOR', 
          'CUSTO BENEFICIO', 
          'DESIGN', 
          'ENTREGA', 
          'FLASH', 
          'FONE', 
          'JOGOS', 
          'MEMÓRIA',
          'PESO', 
          'PREÇO', 
          'PROCESSADOR', 
          'QUALIDADE', 
          'RESISTÊNCIA', 
          'TAMANHO', 
          'TELA', 
          'TRAVAMENTO',
          'VELOCIDADE', 
          'GENÉRICO/OUTRO'
         ]

# cria dicionário associndo um número de 0 a 23 a cada tópico na ordem da lista
dic_topics = {}
for n, item in enumerate(topics):
    dic_topics[n]=item

# tokeniza os tópicos da lista
topics_token = [preprocess_text(item) for item in topics]

# Primeiros labels 

- Criação dos 23 primeiros labels, onde são decididos vendo se o comentário possui a palavra do label, em ordem de prioridade da lista.

In [6]:
# defines empty list
lista_clusters = []

# cria clusters onde cada um deles possui comentários contendo um dos tópicos da lista, caso um comentário contenha dois
# dos tópicos, a ordem de prioridade é a mesma ordem na qual os tópicos estão na lista.
for n, objeto in enumerate(topics_token):
    
    if len(objeto) == 1:
        lista_1 = [item for item in df3.token if objeto[0] in item]
        for n, item in enumerate(lista_clusters):
            lista_1 = [item1 for item1 in lista_1 if item1 not in item]
        lista_clusters.append(lista_1)

    else:
        lista_2 = [item for item in df3.token if (
            objeto[0] in item) and (objeto[1] in item)]
        for n, item in enumerate(lista_clusters):
            lista_2 = [item1 for item1 in lista_2 if item1 not in item]
        lista_clusters.append(lista_2)

# Último Label

- Criação do último label, escolhendo os comentários no dataset que não possui nenhum dos 23 tópicos contidos no seu texto, sendo então caracterizado como "Genérico/outro"

- Temos uma lista onde cada elemento é uma lista com todas os comentários do dataset separados por seu label

- Criação de uma copia da lista, para fazer o balanceamento sem perder informações da lista de clusters original, caso ela seja necessário depois.

- Balanceamento para diminuir os 40 mil comentários na categoria "Genérico/outros" de maneira aleatória.

- Ao final conferimos o tamanho dessa lista, onde cada um desses tamanho é a quantidade de comentários em seu respectivo label.

In [7]:
temp = []

# junta todos os comentários de todos os tópicos em uma única lista sem divisão
for n, objeto in enumerate(lista_clusters):
    temp.extend(objeto)

# cria um último clusters formado por todos os comentários que não possui os elementos da lista acima, isto é não possui
# nenhum dos tópicos
ultimo_cluster = [item for item in df3.token if item not in temp]

# define esse último cluster como último elemento da lista de clusters, que vai servir como o tópico "genérico/outro"
lista_clusters[23] = ultimo_cluster

# cria uma copia da lista_clusters
lista_clusters1 = lista_clusters.copy()

# toma um sample de 2000 comentários para os clusters que tiveram mais que 3500 comentários, diminuindo o balanceamento
for n, item in enumerate(lista_clusters1):
    if len(item) > 3500:
        lista_clusters1[n] = random.sample(item, 2000)    

# Vectorization

 - definindo a função que usaremos para vetorizar os textos

In [8]:
# carrega as arquivo "pt_core_nes_md" do spacy, usado para vetorização de textos em português
nlp = spacy.load('pt_core_news_md')

# define a vetorização usando a função str.vector do spacy
def vec(s):
    return nlp.vocab[s].vector

# Vetorização - Coluna

 - Vetorizando toda a coluna de comentários, transformando cada uma das palavras em um vetor com 300 cordenadas e colocamos a média de cada uma dessas frases em uma matriz final.

In [9]:
# cria uma lista com a quantidade de elementos em cada um dos clusters
linhas = [len(item) for item in lista_clusters1]

# tamanho das palavras vetorizadas
vec_size = 300

# soma a quantidade de elementos de todos os clusters
rows = sum(linhas)

# cria uma lista vazia
list_of_matrix = [] 

# cria uma matriz vazia para ser preenchida com os comentários vetorizados ao final
final_feature_matrix = np.empty([rows, vec_size])

# cria preenche a matriz vazia criada acima com os comentários vetorizados
for n, item in enumerate(lista_clusters1):
    for corpus in item: 
        matrix = np.empty([len(corpus), vec_size]) 
                                              
        for idx, word in enumerate(corpus):
            matrix[idx,:] = vec(word) 
        list_of_matrix.append(matrix)

# cama comentário vetorizado possui 300 variáveis tiramos a média de cada palavra em cada frase para preencher a matriz
# criada acima e preenche-la com esses vetores
for row in range(rows):
    final_feature_matrix[row,:] = list_of_matrix[row].mean(axis = 0)

# Labels e Shuffle

- Adicionando a matriz vetorizada, uma coluna com labels, baseada no tamanho de cada uma das listas contidas na "lista_clusters1"

- Fazendo um shuffle na matriz para diminuir as chances de viés no treinamento do modelo

In [10]:
labels = []

# criamos uma lista com os labels de 0 a 23 para cada tópico
for n, item in enumerate(lista_clusters1):
    for i, objeto in enumerate(item):
        labels.append(n)
    
# transformamos em um array
x = np.array(labels)

# fazemos o reshape para concatenar a matriz de variáveis
x = x.reshape(-1,1)

#  concatenamos a matriz de variáveis com os labeis
final_matrix = np.concatenate((final_feature_matrix, x), axis=1)

# fazemos um shuffle para os comentários não ficar na sequencia pelos labels
final_matrix1 = shuffle(final_matrix, random_state = 42)

# Treinamento e predição

- Dividindo os dados em treino e teste usando a função train_test_split do kslearn

- Treinando um knn classificador com 2 vizinhos

- Fazendo a predição nos dados de teste

- Conferindo acurácia

[Documentação Knn](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

In [11]:
# dividindo em treino e teste
Xtreino, Xteste, ytreino, yteste = train_test_split(
    final_matrix1[:, 0:-1], final_matrix1[:, -1], train_size = 0.7, random_state=42)

# definindo o knn com 2 vizinhos
knn = KNeighborsClassifier(n_neighbors = 2)

# treinando o knn
knn.fit(Xtreino, ytreino)

# fazendo a predição
pred = knn.predict(Xteste)

# vendo a acurácia
accuracy_score(yteste,pred)

0.6102679407907063

# Testando o modelo em frase genérica

- Testamos o modelo na frase genérica "otimo" fazendo todo o pré processamento até a vetorização necessária, para depois aplicar o knn treinando e por fim, usar o dicionario para ver em qual label ele foi classificado.

In [12]:
a = 'Otimo'

# processando o texto
textoProcessado = preprocess_text(a)

# criando a matrix vazia com a quantidade de linhas igual a quantidade de palavras do texto e 300 colunas
matrix = np.empty([len(textoProcessado), 300])

# vetorizado o texto
for idx, word in enumerate(textoProcessado):
    matrix[idx,:] = vec(word)

# criando uma matriz vizinha para preencher com o texto vetorizado
final_feature_matrix = np.empty([1, 300])

# preenchendo a matriz com a média de cada palavra vetorizada
final_feature_matrix = matrix.mean(axis = 0).reshape(1,-1)

# fazendo a predição
knn.predict(final_feature_matrix)

# usando o dicionario criado com os tópicos para ver qual a predição
dic_topics[knn.predict(final_feature_matrix)[0]]

'GENÉRICO/OUTRO'

In [13]:
def classifica_dataframe(df):
    
    df2 = preprocess_data(df, 
                      columns=['KEY','COMMENT_ID','COMMENT_TEXT'],
                      null = True
                     )
    
    df2 = tokenizacao(df2)
    
    for n, item in enumerate(df2.token):
        if len(item)==0:
            df2 = df2.drop(n)
    
    df2 = df2.reset_index().drop(columns={'index'})
    lista_clusters = []
    
    for n, objeto in enumerate(topics_token):
        
        if len(objeto) == 1:
            lista_1 = []
            for i in range(len(df2)):
                if (objeto[0] in df2.token[i]):
                    lista_1.append([df2.token[i], df2.KEY[i]])
            for n, item in enumerate(lista_clusters):
                lista_1 = [item1 for item1 in lista_1 if item1 not in item]
            lista_clusters.append(lista_1)

        else:
            lista_2 = []
            for i in range(len(df2)):
                if (objeto[0] in df2.token[i]) and (objeto[1] in df2.token[i]):
                    lista_2.append([df2.token[i], df2.KEY[i]])
            for n, item in enumerate(lista_clusters):
                lista_2 = [item1 for item1 in lista_2 if item1 not in item]
            lista_clusters.append(lista_2)
            
    temp = []
    for n, objeto in enumerate(lista_clusters[:23]):
        lista_temp = [item[0] for item in objeto] 
        temp.extend(lista_temp)
    ultimo_cluster = []
    for i in range(len(df2)):
        if (df2.token[i] not in temp):
            ultimo_cluster.append([df2.token[i], df2.KEY[i]])

    lista_clusters[23] = ultimo_cluster
    
    labels = []
    
    for n, item in enumerate(lista_clusters):
        for i, objeto in enumerate(item):
            labels.append(n)
    
    categorias = [dic_topics[item] for item in labels]
    
    dff = pd.DataFrame(lista_clusters[0])
    for n, objeto in enumerate(lista_clusters[1:]):
        dff = pd.concat([dff,pd.DataFrame(objeto)])
    
    dff = dff.rename(columns={1:'KEY'})
    
    dff['categoria'] = categorias
    
    df_final = pd.merge(df2,dff, on='KEY')
    df_final = df_final[['KEY','COMMENT_ID','COMMENT_TEXT','categoria']]
    
    return df_final

In [14]:
df3 = classifica_dataframe(df)

In [15]:
df3

,KEY,COMMENT_ID,COMMENT_TEXT,categoria
0,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,9018105,Ótimo,GENÉRICO/OUTRO
1,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,9011802,Ótimo,GENÉRICO/OUTRO
2,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,9011259,Ótimo,GENÉRICO/OUTRO
3,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,8973751,Muito bom,GENÉRICO/OUTRO
4,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,8969185,Ótimo,GENÉRICO/OUTRO
...,...,...,...,...
62061,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc2Ftc3...,9468665,Celular top...não dura 3 dias a bateria...mas ...,BATERIA
62062,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc2Ftc3...,9467820,"Um excelente smartphone, bateria excepcional, ...",BATERIA
62063,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc2Ftc3...,9466851,Até o presente momento tá ótimo.,GENÉRICO/OUTRO
62064,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc2Ftc3...,9465702,Excelente produto!,GENÉRICO/OUTRO
